# Putting many TinyRNNs onto one GPU

Here we investigate a potential speed-up obtained by training many TinyRNNs with the same training data. This means we can run all hyperparameters on the same loop. Could be an upwards of 100x speed up.

In [1]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
from NM_TinyRNN.code.models import gpu_training as gt
from NM_TinyRNN.code.models import datasets as ds
from NM_TinyRNN.code.models import rnns

from pathlib import Path
CODE_DIR = Path('.')
SAVE_PATH = CODE_DIR/'NM_TinyRNN/data/rnns'
DATA_PATH = './NM_TinyRNN/data/AB_behaviour/WS16'

test_path  = './NM_TinyRNN/data/rnns/gpu_test'

trainer = gt.TrainerGPU(weight_seeds = list(range(1,11)),
                        sparsity_lambdas = [1e-4,1e-5,1e-6],
                        energy_lambdas = [0.1,1e-2,1e-3],
                        hebbian_lambdas = [0.0])
model = rnns.TinyRNN(rnn_type = 'vanilla', hidden_size = 2)
dataset = ds.AB_Dataset(DATA_PATH, sequence_length = 100)
final_state_dict, config = trainer.fit(model, dataset)

Parallelizing 90 models on cpu


 30%|██▉       | 299/1000 [00:46<01:49,  6.39it/s]

Search complete. Best model index: 57. Val. loss: 0.5119525194168091


In [3]:
from NM_TinyRNN.code.models import nested_cv as nc
from NM_TinyRNN.code.models import nested_cv_io as save_data

from importlib import reload
reload(save_data)
splits = nc.nested_cv_splits(dataset)
trials_df = save_data.get_model_trial_by_trial_df(model, dataset,splits['inner_folds'][0])
#import seaborn as sns
#sns.scatterplot(trials_df, x='hidden_1',y='hidden_2',hue='logit_value',palette='coolwarm')

In [22]:
reload(gt)
outer_results = nc.run_outer_fold(model, dataset,
                                  outer_loop_number = 1,
                                  n_outer_loops = 10,
                                  save_path = './NM_TinyRNN/data/rnns/gpu_test')
print([d['val_loss'] for d in outer_results['inner_results']])

[outer 1/10]  outer eval: 7 blocks  |  9 inner folds  |  saving to NM_TinyRNN/data/rnns/gpu_test
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu
Parallelizing 160 models on cpu


  0%|          | 0/1000 [00:00<?, ?it/s] 1.64it/s]

Search complete. Best model index: 153. Val. loss: 0.5084373950958252
  Saved outer_fold_1/inner_fold_6 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_6  (val_loss=0.5084, eval_loss=0.4743)
Parallelizing 160 models on cpu


 28%|██▊       | 284/1000 [01:59<05:01,  2.38it/s]


[0.49204152822494507, 0.5087289810180664, 0.4867085814476013, 0.4596385061740875, 0.4951298236846924, 0.5291123390197754, 0.5084373950958252, 0.5372259020805359, 0.4755541682243347]


In [85]:
!squeue -u cburns

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
           2756458      a100     bash   cburns  R    7:30:31      1 gpu-sr670-21
           2756420       cpu     bash   cburns  R    8:58:53      1 enc1-node1
           2759882       cpu NM_TinyR   cburns  R      30:29      1 enc3-node4


In [86]:
nj.get_job_info_df().drop_duplicates(['subject_ID','model_id'])


,subject_ID,outer_loop_n,model_type,hidden_size,nonlinearity,input_encoding,constraint,nm_size,nm_dim,nm_mode,model_id,save_path,data_path,completed
0,WS20,1,vanilla,1,relu,unipolar,energy,1,1,row,1_unit_vanilla_relu_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS20/vanilla/en...,NM_TinyRNN/data/AB_behaviour/WS20,0
1,WS20,1,vanilla,2,relu,unipolar,energy,1,1,row,2_unit_vanilla_relu_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS20/vanilla/en...,NM_TinyRNN/data/AB_behaviour/WS20,0
2,WS20,1,vanilla,1,tanh,unipolar,sparsity,1,1,row,1_unit_vanilla_tanh_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS20/vanilla/sp...,NM_TinyRNN/data/AB_behaviour/WS20,0
3,WS20,1,vanilla,2,tanh,unipolar,sparsity,1,1,row,2_unit_vanilla_tanh_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS20/vanilla/sp...,NM_TinyRNN/data/AB_behaviour/WS20,0
4,WS20,1,monoGRU,1,relu,unipolar,energy,1,1,row,1_unit_monoGRU_relu_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS20/monoGRU/en...,NM_TinyRNN/data/AB_behaviour/WS20,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1771,WS09,1,GRU,2,tanh,unipolar,sparsity,1,1,row,2_unit_GRU_tanh_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS09/GRU/sparsity,NM_TinyRNN/data/AB_behaviour/WS09,0
1772,WS09,1,constGate,1,relu,unipolar,energy,1,1,row,1_unit_constGate_relu_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS09/constGate/...,NM_TinyRNN/data/AB_behaviour/WS09,0
1773,WS09,1,constGate,2,relu,unipolar,energy,1,1,row,2_unit_constGate_relu_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS09/constGate/...,NM_TinyRNN/data/AB_behaviour/WS09,0
1774,WS09,1,constGate,1,tanh,unipolar,sparsity,1,1,row,1_unit_constGate_tanh_unipolar,NM_TinyRNN/data/rnns/nested_DA/WS09/constGate/...,NM_TinyRNN/data/AB_behaviour/WS09,0


In [87]:
from NM_TinyRNN.code.models import nested_jobs as nj
reload(nj)
nj.get_job_info_df().drop_duplicates(['subject_ID','model_id'])
nj.run_training()

Submitting model training for WS20 to HPC
Submitted batch job 2759900
Submitting model training for WS20 to HPC
Submitted batch job 2759901
Submitting model training for WS20 to HPC
Submitted batch job 2759902
Submitting model training for WS20 to HPC
Submitted batch job 2759903
Submitting model training for WS20 to HPC
Submitted batch job 2759904
Submitting model training for WS20 to HPC
Submitted batch job 2759905
Submitting model training for WS20 to HPC
Submitted batch job 2759906
Submitting model training for WS20 to HPC
Submitted batch job 2759907
Submitting model training for WS20 to HPC
Submitted batch job 2759908
Submitting model training for WS20 to HPC
Submitted batch job 2759909
Submitting model training for WS20 to HPC
Submitted batch job 2759910
Submitting model training for WS20 to HPC
Submitted batch job 2759911
Submitting model training for WS20 to HPC
Submitted batch job 2759912
Submitting model training for WS20 to HPC
Submitted batch job 2759913
Submitting model tra

In [88]:
!squeue -u cburns

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
           2756458      a100     bash   cburns  R    7:31:11      1 gpu-sr670-21
           2759911       cpu NM_TinyR   cburns CF       0:09      1 enc1-node12
           2759912       cpu NM_TinyR   cburns CF       0:09      1 enc1-node13
           2759913       cpu NM_TinyR   cburns CF       0:09      1 enc3-node6
           2759914       cpu NM_TinyR   cburns CF       0:09      1 enc3-node7
           2759920       cpu NM_TinyR   cburns PD       0:00      1 (Priority)
           2759921       cpu NM_TinyR   cburns PD       0:00      1 (Priority)
           2759922       cpu NM_TinyR   cburns PD       0:00      1 (Priority)
           2759923       cpu NM_TinyR   cburns PD       0:00      1 (Priority)
           2759924       cpu NM_TinyR   cburns PD       0:00      1 (Priority)
           2759925       cpu NM_TinyR   cburns PD       0:00      1 (Priority)
           2759926       cpu NM_TinyR   cb

In [20]:
[d['val_loss'] for d in outer_results['inner_results']]

[0.5013362169265747,
 0.47611650824546814,
 0.5211803913116455,
 0.520017147064209]